In [1]:
import os
import shutil
import time
import pickle

from concurrent.futures import ThreadPoolExecutor

import numpy as np
import rasterio
import requests

from selenium import webdriver
from selenium.webdriver.common.by import By


In [2]:
class DataConfig:
    BANDS = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
    CLOUD_THRESHOLD = 20
    MIN_VALID_FRACTION = 0.35
    MAX_OUTLIER_FRACTION = 0.20
    MIN_DATES_AFTER_FILTER = 6
    START_INDEX = 347

    def save_config(self, file_path: str):
        config = {
            "BANDS": self.BANDS,
            "CLOUD_THRESHOLD": self.CLOUD_THRESHOLD,
            "MIN_VALID_FRACTION": self.MIN_VALID_FRACTION,
            "MAX_OUTLIER_FRACTION": self.MAX_OUTLIER_FRACTION,
            "MIN_DATES_AFTER_FILTER": self.MIN_DATES_AFTER_FILTER,
            "START_INDEX": self.START_INDEX,
        }
        with open(file_path, "wb") as f:
            pickle.dump(config, f)

    
    def load_config(self, file_path: str):
        with open(file_path, "rb") as f:
            config = pickle.load(f)
        self.BANDS = config["BANDS"]
        self.CLOUD_THRESHOLD = config["CLOUD_THRESHOLD"]
        self.MIN_VALID_FRACTION = config["MIN_VALID_FRACTION"]
        self.MAX_OUTLIER_FRACTION = config["MAX_OUTLIER_FRACTION"]
        self.MIN_DATES_AFTER_FILTER = config["MIN_DATES_AFTER_FILTER"]
        self.START_INDEX = config["START_INDEX"]

data_config = DataConfig()
data_config.load_config("data_config.pkl")
print(data_config.START_INDEX)

347


In [3]:
save_dir = "/home/mohamed-ashraf/Desktop/projects/sat-project/data"

mapping = {
    1: 3,  # Water -> Water
    2: 4,  # Artificial Bare -> Cement
    3: 2,  # Natural Bare -> Sand
    5: 1,  # Woody -> Greenery
    6: 1,  # Cultivated -> Greenery
    7: 1   # Natural Veg -> Greenery
}

tiles = [
    # "28QDE", "29NMG", "29PKL", "29PNM", "30PTV", "30PWT", "31NFH", "31PGR", "31PGS", "32PLR", "32PLS", "32PPA", "32PQQ", "32PRQ", 
    #"33KUT", "33KVU", "33KXV", 
    "33LUC", "33LUD", "33LYE", "33MZM", "33PUQ", "33PWM", "35LMC", "35LNF", "35MNT", "35NLA", 
    "35NRD", "35PNL", "35PNR", "35PRL", "36MWT", "36MWU", "36MZT", 
]


In [4]:
def apply_mapping(label):
    mapped = np.zeros_like(label, dtype=np.uint8)
    for original, new in mapping.items():
        mapped[label == original] = new
    return mapped


def save_mask(input_path, save_path):
    with rasterio.open(input_path) as src:
        img = src.read()
        profile = src.profile

    label = img[0].astype(np.uint8)
    confidence = img[1].astype(np.uint8)
    mapped = apply_mapping(label)

    profile.update(count=2, dtype="uint8")

    with rasterio.open(save_path, "w", **profile) as dst:
        dst.write(mapped, 1)
        dst.write(confidence, 2)

    print(f"Saved mask with confidence: {save_path}")


def download_file(url, path, cookies, timeout=120):
    if os.path.exists(path):
        return path

    with requests.Session() as session:
        for name, value in cookies.items():
            session.cookies.set(name, value)

        response = session.get(url, stream=True, timeout=timeout)
        response.raise_for_status()
        with open(path, "wb") as handle:
            for chunk in response.iter_content(8192):
                if chunk:
                    handle.write(chunk)

    return path


def download_many(download_jobs, cookies, max_workers=6):
    pending_jobs = [(url, path) for url, path in download_jobs if not os.path.exists(path)]
    if not pending_jobs:
        return

    worker_count = min(max_workers, len(pending_jobs))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = [executor.submit(download_file, url, path, cookies) for url, path in pending_jobs]
        for future in futures:
            future.result()


def summarize_scene(masked_img):
    return np.array([
        np.nanmedian(masked_img[band_idx])
        for band_idx in range(masked_img.shape[0])
    ], dtype=np.float32)


def choose_dates(scene_features):
    features = np.stack(scene_features, axis=0)
    center = np.median(features, axis=0)
    scale = np.median(np.abs(features - center), axis=0)
    scale = np.where(scale < 1e-6, 1.0, scale)
    scores = np.mean(np.abs((features - center) / scale), axis=1)

    keep_n = int(np.ceil((1.0 - data_config.MAX_OUTLIER_FRACTION) * len(scene_features)))
    keep_n = max(data_config.MIN_DATES_AFTER_FILTER, keep_n)
    keep_n = min(len(scene_features), keep_n)

    keep_idx = np.argsort(scores)[:keep_n]
    return keep_idx, scores


In [ ]:
for tile_code in tiles:
    for chip_idx in range(30):
        chip_name = f"{chip_idx:02d}"
        chip_url = f"https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/{tile_code}/{chip_name}"
        raw_dir = os.path.join(save_dir, f"raw_{tile_code}_{chip_name}")
        os.makedirs(raw_dir, exist_ok=True)

        driver = webdriver.Chrome()
        try:
            driver.get(chip_url)
            time.sleep(5)

            content_div = None
            retries_left = 25
            while retries_left > 0:
                try:
                    content_div = driver.find_element(
                        By.CSS_SELECTOR,
                        "div.rt-Box[style*='max-height: 800px'][style*='overflow: auto']"
                    )
                    break
                except Exception as exc:
                    retries_left -= 1
                    if retries_left == 0:
                        raise RuntimeError(
                            f"Failed to load content for {tile_code}/{chip_name}. Last error: {exc}"
                        )
                    print(f"Retrying {tile_code}/{chip_name} after load failure: {exc}")
                    driver.refresh()
                    time.sleep(5 if retries_left > 10 else 6)

            cookies = {
                cookie["name"]: cookie["value"]
                for cookie in driver.get_cookies()
            }

            last_height = 0
            while True:
                driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", content_div)
                time.sleep(1)
                new_height = driver.execute_script("return arguments[0].scrollHeight", content_div)
                current_top = driver.execute_script("return arguments[0].scrollTop", content_div)
                if new_height == last_height or current_top + 5 >= new_height - content_div.size["height"]:
                    break
                last_height = new_height

            links = content_div.find_elements(By.CSS_SELECTOR, "a[href]")
            if not links:
                print(f"Empty chip {tile_code}/{chip_name}, skipping.")
                continue

            date_urls = []
            mask_url = None
            mask_name = None

            for anchor in links:
                href = anchor.get_attribute("href")
                fname = anchor.get_attribute("download")

                if not href:
                    continue

                tail = href.rstrip("/").split("/")[-1]
                if tail.startswith(f"{tile_code}_{chip_name}_20") and len(tail) == len(f"{tile_code}_{chip_name}_20180103"):
                    date_urls.append(href.rstrip("/"))

                if fname and fname.endswith("_LC_10m.tif"):
                    mask_url = href
                    mask_name = fname

            date_urls = sorted(set(date_urls))
            print(f"{tile_code}/{chip_name}: found {len(date_urls)} date folders")

            if mask_url is None:
                raise RuntimeError(f"LC mask not found for {tile_code}/{chip_name}")

            os.makedirs(os.path.join(save_dir, "masks"), exist_ok=True)
            os.makedirs(os.path.join(save_dir, "imgs"), exist_ok=True)

            mask_local_path = os.path.join(raw_dir, mask_name)
            download_file(mask_url, mask_local_path, cookies)

            raw_images = []
            masked_images = []
            scene_features = []
            out_profile = None

            for date_url in date_urls:
                print("Processing:", date_url)
                driver.get(date_url)
                time.sleep(2)

                anchors = driver.find_elements(By.CSS_SELECTOR, "a[href]")
                band_jobs = {}
                cld_job = None

                for anchor in anchors:
                    href = anchor.get_attribute("href")
                    fname = anchor.get_attribute("download")

                    if not href or not fname:
                        continue

                    matched_band = False
                    for band_name in data_config.BANDS:
                        if f"_{band_name}_10m.tif" in fname:
                            local_path = os.path.join(raw_dir, fname)
                            band_jobs[band_name] = (href, local_path)
                            matched_band = True
                            break

                    if matched_band:
                        continue

                    if "_CLD_10m.tif" in fname:
                        local_path = os.path.join(raw_dir, fname)
                        cld_job = (href, local_path)

                if len(band_jobs) != len(data_config.BANDS) or cld_job is None:
                    print("Skipped incomplete date")
                    continue

                download_many(
                    [(band_jobs[band_name][0], band_jobs[band_name][1]) for band_name in data_config.BANDS] + [cld_job],
                    cookies,
                )

                band_files = {
                    band_name: band_jobs[band_name][1]
                    for band_name in data_config.BANDS
                }
                cld_file = cld_job[1]

                bands_stack = []
                for band_name in data_config.BANDS:
                    with rasterio.open(band_files[band_name]) as src:
                        bands_stack.append(src.read(1).astype(np.float32))
                        if out_profile is None:
                            out_profile = src.profile.copy()

                raw_img = np.stack(bands_stack, axis=0)

                with rasterio.open(cld_file) as src:
                    cld = src.read(1)

                cloud_mask = cld >= data_config.CLOUD_THRESHOLD
                valid_fraction = 1.0 - float(cloud_mask.mean())
                if valid_fraction < data_config.MIN_VALID_FRACTION:
                    print(f"Skipped low-valid date: {valid_fraction:.2%} valid pixels")
                    continue

                masked_img = raw_img.copy()
                masked_img[:, cloud_mask] = np.nan

                scene_feature = summarize_scene(masked_img)
                if np.isnan(scene_feature).any():
                    print("Skipped unstable date after masking")
                    continue

                raw_images.append(raw_img)
                masked_images.append(masked_img)
                scene_features.append(scene_feature)

            if not masked_images:
                print(f"No valid dates for {tile_code}/{chip_name}, skipping.")
                continue

            keep_idx, scores = choose_dates(scene_features)
            masked_stack = np.stack([masked_images[idx] for idx in keep_idx], axis=0)
            raw_stack = np.stack([raw_images[idx] for idx in keep_idx], axis=0)

            median_img = np.nanmedian(masked_stack, axis=0)
            fallback_img = np.median(raw_stack, axis=0)
            median_img = np.where(np.isnan(median_img), fallback_img, median_img).astype(np.float32)

            out_profile.update(count=len(data_config.BANDS), dtype="float32")
            final_img_path = os.path.join(save_dir, f"imgs/img_{data_config.START_INDEX:02d}.tif")
            final_mask_path = os.path.join(save_dir, f"masks/mask_{data_config.START_INDEX:02d}.tif")

            try:
                with rasterio.open(final_img_path, "w", **out_profile) as dst:
                    dst.write(median_img)
                save_mask(mask_local_path, final_mask_path)
            except Exception:
                for partial_path in (final_img_path, final_mask_path):
                    if os.path.exists(partial_path):
                        os.remove(partial_path)
                raise

            print(f"Kept {len(keep_idx)}/{len(masked_images)} dates after outlier filtering")
            print(f"Worst kept outlier score: {scores[keep_idx].max():.3f}")
            print(f"Saved image: {final_img_path}")
            print(f"Saved mask : {final_mask_path}")

            data_config.START_INDEX += 1
            data_config.save_config("data_config.pkl")
        finally:
            driver.quit()
            shutil.rmtree(raw_dir, ignore_errors=True)


33LUC/00: found 18 date folders
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181015
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181020
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181025
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181030
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181104
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181105
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181109
Processing: https://source.coop/radiantearth/landcovernet/landcovernet_af/data/v1.0/2018/33LUC/00/33LUC_00_20181110
Processing: https://source.coop/radiante

/tmp/ipykernel_380005/1636451271.py:172: RuntimeWarning: All-NaN slice encountered
  median_img = np.nanmedian(masked_stack, axis=0)


Saved mask with confidence: /home/mohamed-ashraf/Desktop/projects/sat-project/data/masks/mask_378.tif
Kept 14/17 dates after outlier filtering
Worst kept outlier score: 1.776
Saved image: /home/mohamed-ashraf/Desktop/projects/sat-project/data/imgs/img_378.tif
Saved mask : /home/mohamed-ashraf/Desktop/projects/sat-project/data/masks/mask_378.tif
Retrying 33LUD/02 after load failure: Message: no such element: Unable to locate element: {"method":"css selector","selector":"div.rt-Box[style*='max-height: 800px'][style*='overflow: auto']"}
  (Session info: chrome=139.0.7258.66); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
#0 0x599286de308a <unknown>
#1 0x599286882a70 <unknown>
#2 0x5992868d4907 <unknown>
#3 0x5992868d4b01 <unknown>
#4 0x599286922d54 <unknown>
#5 0x5992868fa40d <unknown>
#6 0x59928692014f <unknown>
#7 0x5992868fa1b3 <unknown>
#8 0x5992868c659b <unknown>
#9 0x5992868c7

In [ ]:
import matplotlib.pyplot as plt

img_path = "data/imgs/img_32.tif"
mask_path = "data/masks/mask_32.tif"

with rasterio.open(img_path) as src:
    img = src.read()

rgb = img[[3, 2, 1]].transpose(1, 2, 0)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

with rasterio.open(mask_path) as src:
    mask = src.read(1)
    confidence = src.read(2)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.title("RGB Image")
plt.imshow(rgb)

plt.subplot(1, 3, 2)
plt.title("Mask")
plt.imshow(mask, cmap="jet")

plt.subplot(1, 3, 3)
plt.title("Confidence")
plt.imshow(confidence, cmap="magma", vmin=0, vmax=100)

plt.show()
